#### Importing Dependencies

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
# from sklearn.datasets import load_boston
from sklearn.model_selection import train_test_split, RepeatedKFold, cross_val_score
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import StackingRegressor
from matplotlib import pyplot as plt
import kagglehub
from datetime import datetime

pd.set_option('display.max_rows', 100)
pd.set_option("display.max_columns", 100)

d:\Audio Integration\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Download latest version
# path = kagglehub.dataset_download("alexefimik/dubai-real-estate-transactions-dataset")
# print("Path to dataset files:", path)

# Loading dataset into pandas dataframe from csv file
data = pd.read_csv("../datasets/Transactions.csv")

# Discarding Arabic Columns
df = data.loc[:, ~data.columns.isin(data.filter(regex='_ar').columns.tolist())]

df.head(5)

,transaction_id,procedure_id,trans_group_id,trans_group_en,procedure_name_en,instance_date,property_type_id,property_type_en,property_sub_type_id,property_sub_type_en,property_usage_en,reg_type_id,reg_type_en,area_id,area_name_en,building_name_en,project_number,project_name_en,master_project_en,nearest_landmark_en,nearest_metro_en,nearest_mall_en,rooms_en,has_parking,actual_worth,meter_sale_price,rent_value,meter_rent_price,no_of_parties_role_1,no_of_parties_role_2,no_of_parties_role_3
0,1-11-2001-165,11,1,Sales,Sell,24-02-2001,1,Land,NaN,NaN,Commercial,1,Existing Properties,364,Al Wasl,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1350000.0,968.75,NaN,NaN,1.0,1.0,0.0
1,3-9-2004-223,9,3,Gifts,Grant,13-12-2004,4,Villa,NaN,NaN,Commercial,1,Existing Properties,365,Al Hudaiba,NaN,NaN,NaN,NaN,Burj Khalifa,Al Jafiliya Metro Station,Dubai Mall,NaN,0,2790000.0,1614.58,NaN,NaN,1.0,1.0,0.0
2,2-13-1996-119,13,2,Mortgages,Mortgage Registration,12-03-2001,1,Land,NaN,NaN,Commercial,1,Existing Properties,390,Burj Khalifa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,20000000.0,21527.83,NaN,NaN,1.0,1.0,0.0
3,2-14-2005-222,14,2,Mortgages,Modify Mortgage,20-09-2005,2,Building,NaN,NaN,Residential / Commercial,1,Existing Properties,388,Oud Metha,NaN,NaN,NaN,NaN,Dubai International Airport,Oud Metha Metro Station,Dubai Mall,NaN,0,25000000.0,9351.81,NaN,NaN,1.0,1.0,0.0
4,3-9-2012-874,9,3,Gifts,Grant,11-10-2012,4,Villa,NaN,NaN,Residential,1,Existing Properties,276,Al Bada,NaN,NaN,NaN,NaN,Burj Khalifa,Trade Centre Metro Station,Dubai Mall,NaN,0,9000000.0,5839.72,NaN,NaN,1.0,1.0,0.0


In [ ]:
# One Hot Encoding Columns
# trans_group_id (drop trans_group_en because trans_group_id is label encoded form of trans_group_en)
# procedure_id (drop procedure_name_en because procedure_id is label encoded form of procedure_name_en)
# property_type_id (drop property_type_en because property_type_id is label encoded form of property_type_en)

 # converting the string type of 'instance_date' to datetime and get the number years since transaction happened.
df['instance_date'] = pd.to_datetime(df['instance_date'], format="%d-%m-%Y")
today = pd.to_datetime(datetime.today().date())
df.dropna(subset=['instance_date'], inplace=True)
df.loc[:, 'no_of_years_since_transaction'] = (today - df['instance_date']).dt.days / 365.25
df.loc[:, 'no_of_years_since_transaction'] = df['no_of_years_since_transaction'].astype(int)

# drop instance_date column
# df.drop(['instance_date], axis=1, inplace=True)

# property_sub_type_id is the label encoded column of property_sub_type_en but we cannot simply keep the property_sub_type_id only
# because 1/4 of the values are missing. In the start we don't need to fill it as XGboost Regressor will take care of it.
# Need KNN imputation for property_sub_type_id filling later.

# There are only 11 categories in property_usage_en column. We can apply One hot encoding on this column

# reg_type_id (drop reg_type_en because reg_type_id is label encoded form of reg_type_en)

# area_id (drop area_name_en because area_id is label encoded form of area_name_en)
# There are 253 unique values in area_id

# building_name_en have 3365 unique categories and almost 1/3 of data is missing

# project_number (drop project_name_en because project_number is label encoded form of project_name_en)

# master_project_en can be used after encoding

# nearest_landmark_en can be used after encoding

# nearest_metro_en can be used after encoding

# nearest_mall_en can be used after One Hot Encoding

# rooms_en can be used after One Hot Encoding





C:\Users\aafzal03\AppData\Local\Temp\ipykernel_29348\3358904022.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['no_of_years_since_transaction'] = (today - df['instance_date']).dt.days / 365.25
C:\Users\aafzal03\AppData\Local\Temp\ipykernel_29348\3358904022.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['no_of_years_since_transaction'] = df['no_of_years_since_transaction'].astype(int)


In [59]:
df.head(10)

,transaction_id,procedure_id,trans_group_id,trans_group_en,procedure_name_en,instance_date,property_type_id,property_type_en,property_sub_type_id,property_sub_type_en,property_usage_en,reg_type_id,reg_type_en,area_id,area_name_en,building_name_en,project_number,project_name_en,master_project_en,nearest_landmark_en,nearest_metro_en,nearest_mall_en,rooms_en,has_parking,actual_worth,meter_sale_price,rent_value,meter_rent_price,no_of_parties_role_1,no_of_parties_role_2,no_of_parties_role_3,no_of_years_since_transaction
0,1-11-2001-165,11,1,Sales,Sell,2001-02-24,1,Land,NaN,NaN,Commercial,1,Existing Properties,364,Al Wasl,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1350000.0,968.75,NaN,NaN,1.0,1.0,0.0,23
1,3-9-2004-223,9,3,Gifts,Grant,2004-12-13,4,Villa,NaN,NaN,Commercial,1,Existing Properties,365,Al Hudaiba,NaN,NaN,NaN,NaN,Burj Khalifa,Al Jafiliya Metro Station,Dubai Mall,NaN,0,2790000.0,1614.58,NaN,NaN,1.0,1.0,0.0,19
2,2-13-1996-119,13,2,Mortgages,Mortgage Registration,2001-03-12,1,Land,NaN,NaN,Commercial,1,Existing Properties,390,Burj Khalifa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,20000000.0,21527.83,NaN,NaN,1.0,1.0,0.0,23
3,2-14-2005-222,14,2,Mortgages,Modify Mortgage,2005-09-20,2,Building,NaN,NaN,Residential / Commercial,1,Existing Properties,388,Oud Metha,NaN,NaN,NaN,NaN,Dubai International Airport,Oud Metha Metro Station,Dubai Mall,NaN,0,25000000.0,9351.81,NaN,NaN,1.0,1.0,0.0,19
4,3-9-2012-874,9,3,Gifts,Grant,2012-10-11,4,Villa,NaN,NaN,Residential,1,Existing Properties,276,Al Bada,NaN,NaN,NaN,NaN,Burj Khalifa,Trade Centre Metro Station,Dubai Mall,NaN,0,9000000.0,5839.72,NaN,NaN,1.0,1.0,0.0,12
5,1-11-2001-230,11,1,Sales,Sell,2001-03-18,2,Building,NaN,NaN,Residential,1,Existing Properties,271,Al Karama,NaN,NaN,NaN,NaN,Burj Khalifa,ADCB Metro Station,Dubai Mall,NaN,0,850000.0,4021.76,NaN,NaN,5.0,1.0,0.0,23
6,2-13-2005-88,13,2,Mortgages,Mortgage Registration,2005-02-07,2,Building,NaN,NaN,Residential / Commercial,1,Existing Properties,271,Al Karama,NaN,NaN,NaN,NaN,Burj Khalifa,ADCB Metro Station,Dubai Mall,NaN,0,4500000.0,2284.80,NaN,NaN,1.0,1.0,0.0,19
7,3-9-2008-90,9,3,Gifts,Grant,2008-06-12,2,Building,NaN,NaN,Residential / Commercial,1,Existing Properties,367,Al Suq Al Kabeer,NaN,NaN,NaN,NaN,Dubai International Airport,Al Ghubaiba Metro Station,Dubai Mall,NaN,0,50000000.0,52029.68,NaN,NaN,1.0,1.0,0.0,16
8,3-9-2017-2436,9,3,Gifts,Grant,2017-12-19,2,Building,NaN,NaN,Commercial,1,Existing Properties,267,Al Raffa,NaN,NaN,NaN,NaN,Burj Khalifa,Al Ghubaiba Metro Station,Dubai Mall,NaN,0,22499968.0,4843.75,NaN,NaN,1.0,1.0,0.0,6
9,1-11-2002-300031,11,1,Sales,Sell,2002-03-25,1,Land,NaN,NaN,Residential,1,Existing Properties,269,Al Hamriya,NaN,NaN,NaN,NaN,Dubai International Airport,Burjuman Metro Station,Dubai Mall,NaN,0,121194.0,3229.26,NaN,NaN,1.0,6.0,0.0,22


In [ ]:
# print (df['no_of_parties_role_3'].unique())
# df.filter(regex="master_project")

[ 0.  2.  4.  1. nan  6.  3.  5.  8. 14.  9. 12.  7. 13. 17. 11.]


In [46]:
print (df['property_sub_type_id'].isna().sum(), df.shape)

240143 (1047960, 32)


In [ ]:

# Load Boston housing dataset
def get_dataset():
    # Load the dataset
    X, y = data.data, data.target
    return X, y